<a href="https://colab.research.google.com/github/Sampavi01/Advanced_Time_Series_Forecasting/blob/time_series/DL_Baseline_GRU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning - Baseline GRU Model


In [1]:
# --- Core Libraries & Plotting ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import io

# --- Helper for Colab File Upload ---
from google.colab import files

# --- Deep Learning Framework (TensorFlow) ---
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, SimpleRNN, Dense, Dropout, TimeDistributed, Conv1D, MaxPooling1D, Flatten, MultiHeadAttention, LayerNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# --- Scikit-Learn Tools ---
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from scipy.special import inv_boxcox

# --- Plotting Style ---
plt.style.use('seaborn-v0_8-whitegrid')

In [2]:
# ---  Load Data  ---
from google.colab import files
print("Please upload your 'featured_aep_data.csv' file")
uploaded = files.upload()
print("\n✅ File uploaded successfully!")

Please upload your 'featured_aep_data.csv' file


Saving featured_aep_data.csv to featured_aep_data.csv

✅ File uploaded successfully!


In [3]:
# Next, upload the parameters file.
print("\nNext, please upload your 'model_parameters.joblib' file.")
uploaded_params = files.upload()
print(f"\n✅ Uploaded '{list(uploaded_params.keys())}' successfully!")


Next, please upload your 'model_parameters.joblib' file.


Saving model_parameters.joblib to model_parameters.joblib

✅ Uploaded '['model_parameters.joblib']' successfully!


In [4]:
import io
# --- 2. Load the Uploaded Data and Parameters ---
# Load the DataFrame from the uploaded CSV
df_ml = pd.read_csv(io.BytesIO(uploaded['featured_aep_data.csv']), index_col='Datetime', parse_dates=True)
print("\nFeatured DataFrame successfully loaded.")
print("Shape of loaded data:", df_ml.shape)

params = joblib.load( 'model_parameters.joblib')


Featured DataFrame successfully loaded.
Shape of loaded data: (121247, 24)


In [5]:
# --- Unpack Parameters ---
lambda_boxcox = params['lambda_boxcox']
train_end_idx = params['train_end_idx']
val_end_idx = params['val_end_idx']
TARGET_TRANSFORMED = params['target_col_transformed']
TARGET_ORIGINAL = params['target_col_original']
FEATURES = params['feature_columns']

In [6]:
# ---  Recreate Splits ---
# The original dataset started on '2004-10-01 01:00:00'.
# We can calculate how many rows were dropped by comparing the start date
# of our new DataFrame to the original start date.

original_start_date = pd.to_datetime('2004-10-01 01:00:00')
actual_start_date = df_ml.index.min() # The first timestamp in our loaded data

# The difference in hours is the number of rows that were dropped
time_difference = actual_start_date - original_start_date
rows_dropped = int(time_difference.total_seconds() / 3600)

print(f"Calculated that {rows_dropped} rows were dropped by the feature engineering process.")

# Now, adjust the original split indices by this amount
adjusted_train_end = train_end_idx - rows_dropped
adjusted_val_end = val_end_idx - rows_dropped

# Use the adjusted indices to split the new df_ml DataFrame
train_df = df_ml.iloc[:adjusted_train_end]  # Renamed for clarity, like in your original code
val_df = df_ml.iloc[adjusted_train_end:adjusted_val_end]
test_df = df_ml.iloc[adjusted_val_end:]

Calculated that 49 rows were dropped by the feature engineering process.


In [7]:
# --- Scale the Data ---
# Neural networks require input features to be scaled, typically between 0 and 1.
# IMPORTANT: We fit the scaler ONLY on the training data to prevent data leakage.
scaler = MinMaxScaler()
# We scale both features and the target together for easier sequence creation.
train_scaled = scaler.fit_transform(train_df[FEATURES + [TARGET_TRANSFORMED]])
val_scaled = scaler.transform(val_df[FEATURES + [TARGET_TRANSFORMED]])
test_scaled = scaler.transform(test_df[FEATURES + [TARGET_TRANSFORMED]])

print("\n✅ Data is loaded, split, and scaled. Ready for sequence creation.")


✅ Data is loaded, split, and scaled. Ready for sequence creation.


### ** Data Preparation for Sequence Models**

This is the most important new concept for deep learning. RNNs (LSTMs, GRUs) see a **sequence** of past data to predict the future. We must transform our 2D data (rows, features) into 3D data: `(samples, timesteps, features)`.

-   **`timesteps`**: How many hours of past data the model looks at (e.g., the last 7 days). This is our "lookback window."

In [10]:
def create_sequences(data, sequence_length, target_index):
    """Creates sequences of data for time series forecasting."""
    X, y = [], []
    for i in range(len(data) - sequence_length):
        # The input sequence is the window of past data
        X.append(data[i:(i + sequence_length), :])
        # The target is the value of the target variable at the end of the window
        y.append(data[i + sequence_length, target_index])
    return np.array(X), np.array(y)

# --- Define Hyperparameters ---
SEQUENCE_LENGTH = 24 * 7 # Look back at the last 7 days of hourly data (168 hours)
TARGET_INDEX = len(FEATURES) # The target is the last column in our scaled data array

# --- Create the sequences for training, validation, and testing ---
X_train_seq, y_train_seq = create_sequences(train_scaled, SEQUENCE_LENGTH, TARGET_INDEX)
X_val_seq, y_val_seq = create_sequences(val_scaled, SEQUENCE_LENGTH, TARGET_INDEX)
X_test_seq, y_test_seq = create_sequences(test_scaled, SEQUENCE_LENGTH, TARGET_INDEX)

print("\n✅ Setup Complete. Data is ready ")


✅ Setup Complete. Data is ready 


**Build, Train, and Evaluate a Gated Recurrent Unit (GRU) model, a popular and often more efficient alternative to the LSTM**   

In [9]:
# --- 1. Build the GRU Model ---
model = Sequential([
    GRU(64, input_shape=(SEQUENCE_LENGTH, X_train_seq.shape[2])),
    Dropout(0.2),
    Dense(1)
])
model.compile(optimizer='adam', loss='mean_squared_error')
print("--- GRU Model Summary ---")
model.summary()

# --- 2. Train the Model ---
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
print("\n--- Training GRU Model ---")
history = model.fit(X_train_seq, y_train_seq,
                    epochs=50, batch_size=64,
                    validation_data=(X_val_seq, y_val_seq),
                    callbacks=[early_stopping], verbose=1)

# --- 3. Evaluate the Model ---
def evaluate_model(model, X_test, y_test_orig, scaler, lambda_val, seq_len, target_idx):
    preds_scaled = model.predict(X_test)
    dummy_array = np.zeros((len(preds_scaled), scaler.n_features_in_))
    dummy_array[:, target_idx] = preds_scaled.ravel()
    preds_boxcox = scaler.inverse_transform(dummy_array)[:, target_idx]
    preds_orig = inv_boxcox(preds_boxcox, lambda_val)
    true_values = y_test_orig.iloc[seq_len:]
    rmse = np.sqrt(mean_squared_error(true_values, preds_orig))
    return rmse

rmse_result = evaluate_model(model, X_test_seq, test_df[TARGET_ORIGINAL], scaler, lambda_boxcox, SEQUENCE_LENGTH, TARGET_INDEX)
print(f"\n--- Final GRU Model Performance ---")
print(f"Test Set RMSE: {rmse_result:.2f} MW")

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


--- GRU Model Summary ---


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 64)             │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,961 (66.25 KB)

 Trainable params: 16,961 (66.25 KB)

 Non-trainable params: 0 (0.00 B)


--- Training GRU Model ---
Epoch 1/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.0226 - val_loss: 6.3797e-04
Epoch 2/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - loss: 0.0022 - val_loss: 3.7517e-04
Epoch 3/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - loss: 0.0013 - val_loss: 5.4051e-04
Epoch 4/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 20s 9ms/step - loss: 7.9201e-04 - val_loss: 2.5646e-04
Epoch 5/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 21s 9ms/step - loss: 5.5808e-04 - val_loss: 1.9227e-04
Epoch 6/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 20s 9ms/step - loss: 4.5111e-04 - val_loss: 1.7005e-04
Epoch 7/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - loss: 3.8212e-04 - val_loss: 2.8663e-04
Epoch 8/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - loss: 3.5697e-04 - val_loss: 1.4320e-04
Epoch 9/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 20s 9ms/step - loss: 3.2686e-04 - val_loss: 3.9266e-04
Epoch 10/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 20s 9ms/step - loss: 3.3475e-04 - val_loss: 1.8657e-04
Epoch 11/50
